## FOTMOB Notebook Instructions

Run the cells in this exact order:
1. Cell 2: Install compatible packages and create the local services shim for pyfotmob.
2. Cell 3: Reinstall pinned package versions to keep pyfotmob compatible.
3. Cell 4: Import requests.
4. Cell 5: Set the FotMob match URL and extract match_id.
5. Cell 6: Import Team, fetch team data for id 8634, and print the result.

Expected output:
- Cell 6 prints team details.
- If the FotMob API endpoint is unavailable, the output may include a fallback note.

In [32]:
%pip install -q pyfotmob requests==2.31.0 pydantic==1.10.8 python-benedict==0.32.0

# Compatibility shim: PyPI pyfotmob 0.0.3 misses the internal `services` package.
from pathlib import Path

shim_dir = Path("/content/services")
shim_dir.mkdir(parents=True, exist_ok=True)

(shim_dir / "__init__.py").write_text("", encoding="utf-8")
(shim_dir / "data.py").write_text(
    """
import json
import re
import requests


DEFAULT_HEADERS = {
    "User-Agent": "Mozilla/5.0",
    "Accept": "application/json, text/plain, */*",
    "Referer": "https://www.fotmob.com/",
}


def _get_by_dotted_key(data, dotted_key):
    current = data
    for part in dotted_key.split('.'):
        if isinstance(current, dict):
            current = current.get(part)
        else:
            return None
    return current


def _team_fallback_from_html(team_id):
    # Public API endpoints now often return 404 HTML, so parse minimal details from team page.
    team_url = f"https://www.fotmob.com/teams/{team_id}/overview"
    html = requests.get(team_url, headers=DEFAULT_HEADERS, timeout=30).text

    title_match = re.search(r"<title>(.*?)</title>", html, flags=re.IGNORECASE | re.DOTALL)
    title_text = title_match.group(1).strip() if title_match else ""
    team_name = title_text.split(" - ")[0].strip() if title_text else str(team_id)

    return {
        "id": int(team_id),
        "details": {
            "name": team_name,
        },
        "_note": "Returned via HTML fallback because FotMob API endpoint was unavailable.",
    }


def handle_league_match_player_team(entity, url, id, key=None):
    payload = None

    try:
        response = requests.get(url, headers=DEFAULT_HEADERS, timeout=30)
        content_type = response.headers.get("content-type", "")

        if response.ok and "application/json" in content_type.lower():
            payload = response.json()
        elif entity == "team":
            payload = _team_fallback_from_html(id)
        else:
            response.raise_for_status()
    except Exception:
        if entity == "team":
            payload = _team_fallback_from_html(id)
        else:
            raise

    if key:
        return _get_by_dotted_key(payload, key)

    return payload
""".strip()
    + "\n",
    encoding="utf-8",
)

print("Created compatibility shim at /content/services/data.py")

Created compatibility shim at /content/services/data.py


In [33]:
# Keep pyfotmob dependency versions compatible (do not upgrade pydantic to v2 here).
%pip install -q requests==2.31.0 pydantic==1.10.8 python-benedict==0.32.0

In [34]:
# Importing requests library for making HTTP requests.
import requests

In [10]:
url = 'https://www.fotmob.com/matches/manchester-city-vs-arsenal/2rhvv#4506614'
match_id = url.split('#')[-1]

In [35]:
from pyfotmob import Team

team = Team(8634)
print(team.get())

{'id': 8634, 'details': {'name': 'Barcelona'}, '_note': 'Returned via HTML fallback because FotMob API endpoint was unavailable.'}
